# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [2]:
# Load the dataset
df = pd.read_csv('data/AviationData.csv', encoding='latin-1')

# Inspect the dataframe shape (rows, columns)
print(df.shape)

# View the first 5 rows
display(df.head())

# Check data types and non-null counts
df.info()

# Summary statistics
display(df.describe())

(88889, 31)


C:\Users\HP\AppData\Local\Temp\ipykernel_10040\3380428850.py:2: DtypeWarning: Columns (6,7,28) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('data/AviationData.csv', encoding='latin-1')


,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,NaN,...,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007
3,20001218X45448,Accident,LAX96LA321,1977-06-19,"EUREKA, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,IMC,Cruise,Probable Cause,12-09-2000
4,20041105X01764,Accident,CHI79FA064,1979-08-02,"Canton, OH",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,1.0,2.0,NaN,0.0,VMC,Approach,Probable Cause,16-04-1980


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  object 
 1   Investigation.Type      88889 non-null  object 
 2   Accident.Number         88889 non-null  object 
 3   Event.Date              88889 non-null  object 
 4   Location                88837 non-null  object 
 5   Country                 88663 non-null  object 
 6   Latitude                34382 non-null  object 
 7   Longitude               34373 non-null  object 
 8   Airport.Code            50132 non-null  object 
 9   Airport.Name            52704 non-null  object 
 10  Injury.Severity         87889 non-null  object 
 11  Aircraft.damage         85695 non-null  object 
 12  Aircraft.Category       32287 non-null  object 
 13  Registration.Number     87507 non-null  object 
 14  Make                    88826 non-null

,Number.of.Engines,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,82805.000000,77488.000000,76379.000000,76956.000000,82977.000000
mean,1.146585,0.647855,0.279881,0.357061,5.325440
std,0.446510,5.485960,1.544084,2.235625,27.913634
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,1.000000
75%,1.000000,0.000000,0.000000,0.000000,2.000000
max,8.000000,349.000000,161.000000,380.000000,699.000000


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [3]:
# 1. Filter Aircraft Category: Keep only "Airplane"
# We use .str.contains to catch variations like "Airplane" or "airplane"
df_filtered = df[df['Aircraft.Category'].str.contains('Airplane', case=False, na=False)].copy()

# 2. Filter Amateur Built: Client only wants professional builds
# We want to exclude rows where Amateur.Built is 'Yes'
df_filtered = df_filtered[df_filtered['Amateur.Built'] != 'Yes']

# 3. Filter Event Date: Keep only events from 1983 onwards
# First, convert the column to datetime format
df_filtered['Event.Date'] = pd.to_datetime(df_filtered['Event.Date'], errors='coerce')

# Now filter for years >= 1983
df_filtered = df_filtered[df_filtered['Event.Date'].dt.year >= 1983]

# Let's check how many rows we have left after filtering
print(f"Rows remaining after filtering: {len(df_filtered)}")

Rows remaining after filtering: 21463


### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [4]:
# --- Metric 1: Serious/Fatal Injury Fraction ---

# First, fill NaN values in injury columns with 0 so math works
injury_cols = ['Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured']
df_filtered[injury_cols] = df_filtered[injury_cols].fillna(0)

# Create an estimated "Total Passengers" by adding up all the injury categories
df_filtered['Total.Passengers_Est'] = df_filtered[injury_cols].sum(axis=1)

# Create a combined column for fatal + serious injuries
df_filtered['Total.Fatal.Serious'] = df_filtered['Total.Fatal.Injuries'] + df_filtered['Total.Serious.Injuries']

# Calculate the fraction. 
# We use np.where to prevent division by zero errors if estimated passengers is 0
df_filtered['Injury_Fraction'] = np.where(
    df_filtered['Total.Passengers_Est'] > 0, 
    df_filtered['Total.Fatal.Serious'] / df_filtered['Total.Passengers_Est'], 
    0 # If 0 passengers, set fraction to 0
)

# --- Metric 2: Aircraft Destroyed ---
df_filtered['Is_Destroyed'] = np.where(
    df_filtered['Aircraft.damage'].str.lower() == 'destroyed', 1, 0
)

# Let's look at the new columns we created to make sure it worked
display(df_filtered[['Total.Passengers_Est', 'Total.Fatal.Serious', 'Injury_Fraction', 'Aircraft.damage', 'Is_Destroyed']].head())

,Total.Passengers_Est,Total.Fatal.Serious,Injury_Fraction,Aircraft.damage,Is_Destroyed
4149,588.0,0.0,0.0,Minor,0
4150,588.0,0.0,0.0,Minor,0
4171,2.0,2.0,1.0,Destroyed,1
4285,5.0,1.0,0.2,NaN,0
5957,289.0,0.0,0.0,Minor,0


**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [5]:
# Clean the Aircraft.damage column to standardize text
# Fill NaNs with 'Unknown' first
df_filtered['Aircraft.damage'] = df_filtered['Aircraft.damage'].fillna('UNKNOWN')

# Standardize capitalization (e.g., 'destroyed' -> 'DESTROYED')
df_filtered['Aircraft.damage'] = df_filtered['Aircraft.damage'].str.upper().str.strip()

# Let's see the unique values in this column now
print("Unique values in Aircraft.damage:")
print(df_filtered['Aircraft.damage'].value_counts())

# --- Now for the Make column (the next section) ---

# 1. Clean the Make column (uppercase and remove spaces)
df_filtered['Make'] = df_filtered['Make'].str.upper().str.strip()

# 2. Keep Makes with a reasonable number (Threshold = 50)
make_counts = df_filtered['Make'].value_counts()
valid_makes = make_counts[make_counts >= 50].index

# Filter the dataframe
df_filtered = df_filtered[df_filtered['Make'].isin(valid_makes)]

print(f"\nRows remaining after filtering Makes: {len(df_filtered)}")

# Show the top 10 most common Makes left in the dataset
display(df_filtered['Make'].value_counts().head(10))

Unique values in Aircraft.damage:
Aircraft.damage
SUBSTANTIAL    16993
DESTROYED       2319
UNKNOWN         1328
MINOR            823
Name: count, dtype: int64

Rows remaining after filtering Makes: 17905


Make
CESSNA                7148
PIPER                 3992
BEECH                 1432
BOEING                1270
MOONEY                 363
AIRBUS                 244
CIRRUS DESIGN CORP     220
BELLANCA               219
AIR TRACTOR INC        219
MAULE                  215
Name: count, dtype: int64

### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

*Make Column Cleaning Tasks:*
1. *Standardization:* Converted all text in the Make column to uppercase and stripped trailing/leading whitespaces to ensure data consistency (e.g., "Cessna" and "CESSNA" become "CESSNA").
2. *Filtering:* Filtered the dataset to only include aircraft makes that have a minimum of 50 accident records. This ensures we have a statistically significant sample size for our analysis.

In [6]:
# 1. Clean the Make column (uppercase and remove spaces)
df_filtered['Make'] = df_filtered['Make'].str.upper().str.strip()

# 2. Keep Makes with a reasonable number (Threshold = 50)
make_counts = df_filtered['Make'].value_counts()
valid_makes = make_counts[make_counts >= 50].index

# Filter the dataframe
df_filtered = df_filtered[df_filtered['Make'].isin(valid_makes)]

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [7]:
# 1. Get rid of any NaNs in the Model column
df_filtered = df_filtered.dropna(subset=['Model'])

# 2. Clean the Model text (uppercase and remove spaces)
df_filtered['Model'] = df_filtered['Model'].str.upper().str.strip()

# 3. Create a derived column that is a unique identifier for a given plane type
df_filtered['Make_Model'] = df_filtered['Make'] + " " + df_filtered['Model']

# Let's verify it worked
display(df_filtered[['Make', 'Model', 'Make_Model']].head())

,Make,Model,Make_Model
4150,BOEING,747,BOEING 747
4171,PIPER,PA-28-140,PIPER PA-28-140
4285,DE HAVILLAND,DHC-6,DE HAVILLAND DHC-6
6760,BOEING,727-200,BOEING 727-200
6806,BEECH,C35,BEECH C35


### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [8]:
# List of columns we want to clean
cols_to_clean = ['Engine.Type', 'Weather.Condition', 'Purpose.of.flight', 'Broad.phase.of.flight']

# Loop through and standardize text (uppercase, strip spaces)
for col in cols_to_clean:
    # Convert to string first just in case there are numbers, then upper and strip
    df_filtered[col] = df_filtered[col].astype(str).str.upper().str.strip()
    # Replace placeholder 'NAN' strings with actual missing values
    df_filtered[col] = df_filtered[col].replace('NAN', np.nan)

# For Number.of.Engines, make sure it's numeric
df_filtered['Number.of.Engines'] = pd.to_numeric(df_filtered['Number.of.Engines'], errors='coerce')

# Let's look at the unique values for a couple of these to make sure it worked
print("Weather Conditions:")
print(df_filtered['Weather.Condition'].value_counts(dropna=False).head())

Weather Conditions:
Weather.Condition
VMC    14298
NaN     2425
IMC      907
UNK      262
Name: count, dtype: int64


### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [9]:
# Get a count of non-null values for each column
non_null_counts = df_filtered.count()

# Create a list of columns to drop that have 5,000 or fewer non-null values
# (Changed from 20,000 to 5,000 to accommodate our smaller filtered dataset)
cols_to_drop = non_null_counts[non_null_counts <= 5000].index

# Drop those columns from the dataframe
df_filtered = df_filtered.drop(columns=cols_to_drop)

print(f"Dropped {len(cols_to_drop)} columns.")
print(f"New dataframe shape: {df_filtered.shape}")

Dropped 2 columns.
New dataframe shape: (17892, 34)


### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [10]:
df_filtered.to_csv('data/Cleaned_Aviation_Data.csv', index=False)
print("Data successfully saved to data/Cleaned_Aviation_Data.csv!")

Data successfully saved to data/Cleaned_Aviation_Data.csv!
